<a href="https://colab.research.google.com/github/gaegoori/5th-diet-ai/blob/finetuning/Unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# 1. Unsloth 설치
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Xformers 및 필수 라이브러리 설치 (버전 제약 제거!)
# "<0.0.27" 같은 숫자를 지워서, 자동으로 최신 Wheel을 받아오게 함
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
print("Unsloth & Xformers 설치 완료! 🚀")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth & Xformers 설치 완료! 🚀


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # 아까 원하셨던 긴 문맥 유지!
dtype = None # None으로 두면 Unsloth가 알아서 L4에 맞게(bf16) 잡습니다.
load_in_4bit = True # 4비트 양자화 (메모리 절약)

# 1. 모델과 토크나이저 불러오기 (이미 4비트된 버전을 가져와서 더 빠름)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", # Unsloth가 최적화해둔 모델
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. LoRA 어댑터 붙이기 (설정값 동일하게 유지)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Unsloth는 0을 권장 (속도 최적화)
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Unsloth 전용 체크포인팅 (메모리 30% 절약)
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

==((====))==  Unsloth 2026.1.4: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth 2026.1.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
from datasets import Dataset

# 1. 원본 파일 경로 (업로드하신 파일명 그대로 사용)
file_path = "drive/MyDrive/integrated_final_finetuning_dataset.jsonl"

# 2. 파이썬 제너레이터 함수 정의
# (파일을 한 줄씩 직접 읽어서 문제가 없는 데이터만 골라냅니다)
def jsonl_generator():
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                # 파이썬의 json 라이브러리는 조금 더 관대합니다.
                data = json.loads(line)

                # 'messages' 키가 있는지 확인하고 반환
                if "messages" in data:
                    yield data

            except json.JSONDecodeError:
                # 에러가 난 줄은 그냥 조용히 건너뜁니다.
                continue

# 3. 제너레이터로부터 바로 데이터셋 생성 (★ 에러가 발생하지 않는 핵심!)
print("데이터셋을 생성 중입니다... (불량 데이터는 자동으로 건너뜁니다)")
dataset = Dataset.from_generator(jsonl_generator)

print(f"✅ 데이터 로드 성공! 총 데이터 개수: {len(dataset)}개")

# 4. (필요시) 채팅 템플릿 적용 등 후속 작업 진행
# Unsloth나 HuggingFace Trainer에서 바로 사용 가능합니다.

데이터셋을 생성 중입니다... (불량 데이터는 자동으로 건너뜁니다)
✅ 데이터 로드 성공! 총 데이터 개수: 19380개


In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template

# 1. 모델과 토크나이저 불러오기 (Qwen 2.5-3B, 4비트)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. 모델에 LoRA 어댑터 장착 (학습할 부분만 붙이기)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 메모리 30% 절약!
    random_state = 3407,
)

# 3. 데이터셋을 학습 가능한 형태로 변환 (ChatML 포맷)
# ★ 방금 로드한 'dataset' 변수를 여기서 사용합니다!
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

print("데이터 변환 중... (잠시만 기다려주세요)")
train_dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. 학습 시작 설정 (L4 최적화: 절대 안 튕기는 설정)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,   # 메모리 안전하게 1
        gradient_accumulation_steps = 64,  # 대신 64번 참았다가 학습 (총 배치 64 효과)
        warmup_steps = 5,
        max_steps = 1000,                  # 1000번만 훈련하고 종료 (속도 최적화)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), # L4는 bf16 자동 켜짐 (속도 UP)
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 5. 진짜 시작!
print("🔥 학습을 시작합니다! 아래 Loss 값이 떨어지는지 확인하세요.")
trainer_stats = trainer.train()

==((====))==  Unsloth 2026.1.4: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
데이터 변환 중... (잠시만 기다려주세요)


Map:   0%|          | 0/19380 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/19380 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


🔥 학습을 시작합니다! 아래 Loss 값이 떨어지는지 확인하세요.


...

wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,1.806900
2,1.751200
3,1.757700
4,1.757100
5,1.682900
6,1.670900
7,1.588400
8,1.624400
9,1.559900
10,1.522600


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


train/epoch,▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇████
train/global_step,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇█
train/grad_norm,█▄▁▂▂▄▄▅▄▃▃▃▃▃▃▃▃▃▃▃▄▄▃▃▄▃▄▃▄▄▄▄▄▄▄▄▄▄▄▄
train/learning_rate,██▇▇▇▇▇▇▇▇▇▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/loss,█▄▄▄▄▃▃▃▃▂▃▂▂▂▂▂▃▂▂▂▂▁▂▁▂▂▁▂▂▂▁▂▁▁▁▂▂▁▂▁
total_flos,1.1015457683786711e+18
train/epoch,3.30052
train/global_step,1000
train/grad_norm,0.19642
train/learning_rate,0.0
train/loss,0.6703


In [ ]:
# 구글 드라이브 경로 지정 (폴더가 없으면 알아서 만듭니다)
save_path = "/content/drive/MyDrive/My-AI-Project/Qwen2.5-3B-Legal-LoRA"

# 모델 저장 (LoRA 어댑터만 저장해서 용량이 작음)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ 모델 저장이 완료되었습니다! 경로: {save_path}")

✅ 모델 저장이 완료되었습니다! 경로: /content/drive/MyDrive/My-AI-Project/Qwen2.5-3B-Legal-LoRA


In [ ]:
from unsloth import FastLanguageModel

# 1. 추론 모드로 전환
FastLanguageModel.for_inference(model)

# =================================================================
# [★ 추가된 해결 코드] 이 부분이 경고를 없애줍니다.
# =================================================================
# "빈칸(Pad)"과 "문장끝(EOS)"이 같다면 명확하게 지정해줌
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델 설정에도 ID를 강제로 입력
model.config.pad_token_id = tokenizer.pad_token_id

# ★ 중요: 추론(생성)할 때는 패딩을 '왼쪽'에 줘야 글이 안 끊깁니다.
tokenizer.padding_side = "left"
# =================================================================


# 2. 질문 만들기
question = "소방시설공사업법에 따른 감리업자가 감리원을 배치할 때 변경 신고는 어떻게 해야 하나요?"

messages = [
    {"role": "system", "content": "당신은 헌법재판소 결정례를 분석하는 법률 전문가입니다."},
    {"role": "user", "content": question},
]

# 3. 입력값 변환 (attention_mask를 자동으로 생성하게 됨)
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. 답변 생성
# (attention_mask 오류가 사라지고 안정적으로 생성됩니다)
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.3,
    pad_token_id = tokenizer.pad_token_id, # 여기도 명시해주면 더 확실함
    eos_token_id = tokenizer.eos_token_id,
)

# 5. 결과 출력
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
# 답변 부분만 잘라서 출력
print("🤖 모델의 답변:\n")
# 전체 텍스트에서 system/user 프롬프트 제거하고 답변만 보려는 시도
# (skip_special_tokens=True를 했으므로 텍스트가 깔끔하게 나옵니다)
print(decoded_output[0].split("assistant")[-1].strip())

🤖 모델의 답변:

1. 법령의 근거 : 이 사건은 소방시설공사업법 제20조 및 같은 법 시행규칙 제24조에 따라 소방시설공사감리업자가 감리원을 배치할 때 변경 신고를 하기 위한 절차와 방법에 관한 것입니다.

2. 사안 요지 : 소방시설공사업법에 따른 감리업자가 감리원을 배치할 때 변경 신고를 하려면, 해당 업체가 소속된 감리원 명단을 작성하여 시·도지사에게 변경 신고해야 합니다. 또한, 변경 신고를 하려면 감리원의 성명, 주민등록번호, 직위, 등록번호 등을 기재한 서류를 제출해야 하며, 시·도지사는 변경 신고를 받은 후 이를 검토하여 적합한지 여부를 확인해야 합니다.

3. 판단 이유 : 소방시설공사업법 제20조에 따르면, 감리업자는 감리원을 배치하거나 변경할 때에는 시·도지사에게 변경 신고를 하도록 규정되어 있습니다. 이는 감리업자가 감리원을 배치하거나 변경할 때의 관계기관과의 협력을 강화하고, 감리원의 적정한 배치와 관리를 통해 공사의 질적 수준을 높이기 위한 것입니다. 따라서, 감리업자가 감리원을 배치하거나 변경할 때에는 반드시 시·도지사에게 변경 신고를 하여야 하고, 그 신고를 받은 시·도지사는 감리원의 적정한 배치 여부를 검토해야 합니다. 이러한 절차와 방법은 감리업자의 감리원 배치나 변경을 정상적으로 진행시키는 데 필요한 것이므로, 감리업자가 이를 위반하면 감리업자의 자격이나 감리업의 질적 수준에 영향을 미칠 수 있습니다. 따라서, 소방시설공사업법에 따른 감리업자가 감리원을 배치하거나 변경할 때에는 반드시 시·도지사에게 변경 신고를 하여야 하며, 시·도지사는 이를 적절히 검토해야 합니다.


In [ ]:


 008 # GGUF 변환 및 구글 드라이브 저장 (q4_k_m: 용량과 성능 밸런스 최고)
gguf_path = "/content/drive/MyDrive/My-AI-Project/Qwen2.5-3B-Legal-GGUF"

print("GGUF 변환을 시작합니다... (약 5~10분 소요)")
model.save_pretrained_gguf(
    gguf_path,
    tokenizer,
    quantization_method = "q4_k_m"
)

print(f"✅ 변환 완료! 이제 '{gguf_path}' 폴더의 파일을 다운로드해서 Ollama에서 쓰세요!")

GGUF 변환을 시작합니다... (약 5~10분 소요)
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 4029.11it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:28<00:00, 14.16s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/My-AI-Project/Qwen2.5-3B-Legal-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Qwen2.5-3B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['Qwen2.5-3B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLM

In [ ]:
# 1. Colab 내부 공간에 GGUF 저장 (드라이브 용량 안 씀)
local_gguf_path = "model.gguf" # 파일명

print("GGUF 변환을 시작합니다... (Colab 임시 공간 사용)")
model.save_pretrained_gguf(
    local_gguf_path,
    tokenizer,
    quantization_method = "q4_k_m"
)
print("✅ 변환 완료!")

GGUF 변환을 시작합니다... (Colab 임시 공간 사용)
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:19<00:00,  9.92s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:20<00:00, 10.17s/it]


Unsloth: Merge process complete. Saved to `/content/model.gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Qwen2.5-3B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['Qwen2.5-3B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: llama-cli --model Qwen2.5-3B-Instruc